In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import collections
import itertools
import tqdm
from copy import deepcopy
from functools import lru_cache

In [ ]:
np.random.seed(0)

In [ ]:
def bayesian_update(priors):
    if np.sum(priors) == 0:
        return np.zeros_like(priors)
    return priors / np.sum(priors)

In [ ]:
def best_response(x, thresholds, priors, c):
    posteriors = bayesian_update(priors)
    search_space = [x] + thresholds[thresholds > x].tolist()
    utilities = []
    for x_p in search_space:
        utility = np.dot(posteriors, x_p >= thresholds)
        cost = c * abs(x-x_p)
        utilities.append(utility - cost)
    return search_space[np.argmax(utilities)]

In [ ]:
def best_response_vectorized(X, thresholds, priors, c):
    posteriors = bayesian_update(priors)

    thresholds = np.array(thresholds)
    X = np.array(X)

    utilities_cumsum = np.array([
        np.dot(posteriors, thresholds[j] >= thresholds)
        for j in range(len(thresholds))
    ])

    pass_matrix = X[:, None] >= thresholds[None, :]
    utility_stay = pass_matrix @ posteriors

    X_col = X[:, None]
    thresholds_row = thresholds[None, :]

    feasible = thresholds_row > X_col
    utility_jump = utilities_cumsum[None, :] - c * np.abs(thresholds_row - X_col)
    utility_jump[~feasible] = -np.inf

    utilities = np.concatenate([utility_stay[:, None], utility_jump],axis=1)

    best_idx = np.argmax(utilities, axis=1)
    X_p = np.where(best_idx == 0, X, thresholds[best_idx - 1])

    return X_p

In [ ]:
def merge_classifiers(X, X_p, thresholds, priors):
    merged_thresholds, merged_priors = deepcopy(thresholds), deepcopy(priors)
    for _ in range(len(thresholds)):
        support = np.array([np.mean(X_p==threshold) for threshold in merged_thresholds])
        dominated = np.where(support == 0)[0]

        if len(dominated) == 0:
            break
        
        for i in reversed(dominated):
            threshold_i = merged_thresholds[i]

            jumps = X_p[X < threshold_i]
            if len(jumps) == 0:
                continue

            candidates = jumps[jumps > threshold_i]
            if len(candidates) == 0:
                continue

            threshold_dom = np.min(candidates)
            dom_idx = np.where(threshold_dom == merged_thresholds)[0][0]

            merged_priors[dom_idx] += merged_priors[i]
            merged_priors = np.delete(merged_priors, i)
            merged_thresholds = np.delete(merged_thresholds, i)

    return merged_thresholds, merged_priors

In [ ]:
def balance_priors(priors, random=True):
    total = np.sum(priors)
    if total == 1:
        return priors
    indices = priors == 0
    remainder = 1 - total
    if random:
        p = np.random.rand(indices.sum())
        p = (p / p.sum()) * remainder
    else:
        p = remainder / indices.sum()
    priors[indices] = p
    return priors

In [ ]:
def accuracy_loss(X, X_p, thresholds, priors, threshold_true):
    Y_true = (X >= threshold_true).astype(float)
    losses = np.empty_like(thresholds)
    for i, threshold in enumerate(thresholds):
        Y_p = (X_p >= threshold).astype(float)
        losses[i] = np.abs(Y_true - Y_p).mean()
    posteriors = bayesian_update(priors)
    return np.dot(losses, posteriors)

In [ ]:
def evaluate_partition(X, partition, thresholds, priors, threshold_true, c, return_all=False):
    thresholds_p = thresholds[partition]
    priors_p = priors[partition]
    X_p = best_response_vectorized(X, thresholds_p, priors_p, c)
    thresholds_p, priors_p = merge_classifiers(X, X_p, thresholds_p, priors_p)
    acc_loss_p = accuracy_loss(X, X_p, thresholds_p, priors_p, threshold_true)
    if return_all:
        return acc_loss_p, thresholds_p, priors_p
    return acc_loss_p

def evaluate_system(X, partitions, thresholds, priors, threshold_true, c):
    acc_loss = 0.
    for partition in partitions:
        acc_loss_p = evaluate_partition(X, partition, thresholds, priors, threshold_true, c)
        acc_loss += acc_loss_p * np.sum(priors[partition])
    return acc_loss

In [ ]:
def set_partitions(collection):
    if len(collection) == 1:
        yield [collection]
        return

    first = collection[0]
    for smaller in set_partitions(collection[1:]):
        for i in range(len(smaller)):
            yield smaller[:i] + [[first] + smaller[i]] + smaller[i+1:]
        yield [[first]] + smaller

In [ ]:
def display_queue(Q, P):
    res = "[  "
    for (a_id, b_id) in Q:
        res += f"({P[a_id]}, {P[b_id]})  "
    res += "]"
    print(res)

def find_partitions_greedy(X, thresholds, priors, threshold_true, c, display=False):
    P = {}
    next_id = 0
    partitions = [[i] for i in range(len(priors))]
    for block in partitions:
        P[next_id] = list(block)
        next_id += 1


    Q = collections.deque(itertools.combinations(P.keys(), 2))

    while Q:
        if display:
            display_queue(Q, P)
        a_id, b_id = Q.popleft()
        if a_id not in P.keys() or b_id not in P.keys():
            continue

        a = P[a_id]
        b = P[b_id]

        acc_loss_a = evaluate_partition(X, a, thresholds, priors, threshold_true, c)
        acc_loss_b = evaluate_partition(X, b, thresholds, priors, threshold_true, c)
        lhs = acc_loss_a * np.sum(priors[a]) + acc_loss_b * np.sum(priors[b])

        ab = sorted(a + b)
        acc_loss_ab = evaluate_partition(X, ab, thresholds, priors, threshold_true, c)
        rhs = acc_loss_ab * np.sum(priors[ab])

        if lhs - rhs > -1e-6:
            del P[a_id]
            del P[b_id]

            Q = collections.deque(
                (x, y)
                for (x, y) in Q
                if x not in {a_id, b_id} and y not in {a_id, b_id}
            )

            new_id = next_id
            next_id += 1
            P[new_id] = ab

            for c_id in P.keys():
                if c_id != new_id:
                    Q.append((new_id, c_id))

            # Q = collections.deque(sorted(Q, key=lambda x: P[x[0]]))
    return list(P.values())

In [ ]:
# def find_partitions_optimal(X, thresholds, priors, threshold_true, c):
#     indices = [i for i in range(len(thresholds))]
#     partitions_set = list(set_partitions(indices))

#     best_partition = None
#     best_loss = np.inf

#     for partitions in tqdm.tqdm(partitions_set):
#         acc_loss = 0.
#         for partition in partitions:
#             acc_loss_p = evaluate_partition(X, partition, thresholds, priors, threshold_true, c)
#             acc_loss += acc_loss_p * np.sum(priors[partition])

#         if acc_loss < best_loss:
#             best_loss = acc_loss
#             best_partition = deepcopy(partitions)

#     return best_partition

In [ ]:
def find_partitions_optimal(X, thresholds, priors, threshold_true, c):
    indices = list(range(len(thresholds)))
    partitions_set = list(set_partitions(indices))

    @lru_cache(maxsize=None)
    def evaluate_partition_cached(partition_tuple):
        partition = list(partition_tuple)
        acc_loss_p = evaluate_partition(
            X, partition, thresholds, priors, threshold_true, c
        )
        return acc_loss_p * np.sum(priors[partition])

    best_partition = None
    best_loss = np.inf

    for partitions in partitions_set:
        acc_loss = 0.0
        for partition in partitions:
            partition_tuple = tuple(sorted(partition))
            acc_loss += evaluate_partition_cached(partition_tuple)

        if acc_loss < best_loss:
            best_loss = acc_loss
            best_partition = deepcopy(partitions)

    return best_partition

In [ ]:
c = 0.5
threshold_true = 0.9999999

threshold_min, threshold_max, threshold_delta = 0., 1., 0.1
# thresholds = np.arange(threshold_min+threshold_delta, threshold_max, threshold_delta).round(4)
X = np.arange(threshold_min, threshold_max+1e-4, 1e-4).round(4)

# # priors = np.zeros_like(thresholds)
# priors = np.array([0.11393634, 0.11459784, 0.03594993, 0.25, 0.02203078, 0.05390004, 0.06215049, 0.09743458, 0.25])
# balance_priors(priors, random=True)

thresholds = np.array([0.116597, 0.218254, 0.283076, 0.315294, 0.38091 , 0.39711 , 0.624227, 0.649651, 0.924501, 0.960969])
priors = np.array([0.026416, 0.130027, 0.15677 , 0.192578, 0.074827, 0.141934, 0.106615, 0.026666, 0.033143, 0.111024])

print(np.sum(priors))
pd.DataFrame({"threshold": thresholds, "priors": priors}).round(6).T

In [ ]:
partition_greedy = find_partitions_greedy(X, thresholds, priors, threshold_true, c)
partition_optimal = find_partitions_optimal(X, thresholds, priors, threshold_true, c)

acc_loss_greedy = evaluate_system(X, partition_greedy, thresholds, priors, threshold_true, c)
acc_loss_optimal = evaluate_system(X, partition_optimal, thresholds, priors, threshold_true, c)

In [ ]:
print("Greedy")
print("------")
print(f"Partition: {sorted(partition_greedy)}")
print(f"Acc Loss : {acc_loss_greedy}")
print()
print("Optimal")
print("-------")
print(f"Partition: {sorted(partition_optimal)}")
print(f"Acc Loss : {acc_loss_optimal}")
print()
print(f"Ratio: {(1 - acc_loss_optimal) / (1 - acc_loss_greedy):.4f}")

In [ ]:
a = [0,1,2,3,4,5,6,7,8]
# X = np.arange(threshold_min, threshold_max, 1e-4).round(4)
X_p = best_response_vectorized(X, thresholds[a], priors[a], c)
px.scatter(x=X, y=X_p)

In [ ]:
a, b = [0,1,2,3,4,5,6,7,8], [9]

acc_loss_a, thresholds_a, priors_a = evaluate_partition(X, a, thresholds, priors, threshold_true, c, True)
acc_loss_b, thresholds_b, priors_b = evaluate_partition(X, b, thresholds, priors, threshold_true, c, True)

lhs = acc_loss_a * np.sum(priors[a]) + acc_loss_b * np.sum(priors[b])
ab = sorted(a + b)

acc_loss_ab, thresholds_ab, priors_ab = evaluate_partition(X, ab, thresholds, priors, threshold_true, c, True)
rhs = acc_loss_ab * np.sum(priors[ab])

print(f"    threshold true: {threshold_true}")
print(f"                 c: {c}")
print(f"                 a: {a}")
print(f"                 b: {b}")
print(f"   accuracy loss a: {acc_loss_a:.7f}")
print(f"   accuracy loss b: {acc_loss_b:.7f}")
print(f"  accuracy loss ab: {acc_loss_ab:.7f}")
print(f"               LHS: {lhs:.7f}")
print(f"               RHS: {rhs:.7f}")
print(f"            merge?: {lhs - rhs > 0}")
print()
print(f"      thresholds a: {thresholds_a}")
print(f"          priors a: {priors_a}")
print(f"      thresholds b: {thresholds_b}")
print(f"          priors b: {priors_b}")
print(f"     thresholds ab: {thresholds_ab}")
print(f"         priors ab: {priors_ab}")

In [ ]:
runs = 100
N = np.arange(2, 11)
# C = np.arange(0., 10, 0.5) + 0.5
C = [0.5]

results = {"n": [], "run": [], "thresholds": [], "priors": [], "threshold_true": [], "c": [], "opt": [], "greedy": [], "acc_loss_opt": [], "acc_loss_greedy": [], "ratio": []}

for n in N:
    for run in tqdm.trange(runs, desc=f"[ n={n} ]"):
        thresholds = np.sort(np.random.rand(n)).round(6)
        priors = np.zeros_like(thresholds)
        balance_priors(priors, random=True)
        diff = 1 - np.sum(priors.round(6))
        priors = priors.round(6)
        priors[0] += diff
        # if abs(1-np.sum(priors)) > 1e-6:
        #     print(np.sum(priors))
        #     continue
        
        # threshold_true = np.random.rand()
        threshold_true = 0.999999
        for c in C:
            partition_opt = find_partitions_optimal(X, thresholds, priors, threshold_true, c)
            partition_greedy = find_partitions_greedy(X, thresholds, priors, threshold_true, c)

            acc_loss_opt = evaluate_system(X, partition_opt, thresholds, priors, threshold_true, c)
            acc_loss_greedy = evaluate_system(X, partition_greedy, thresholds, priors, threshold_true, c)

            ratio = (1 - acc_loss_opt)/(1- acc_loss_greedy)

            results["n"].append(n)
            results["run"].append(run)
            results["thresholds"].append(deepcopy(thresholds))
            results["priors"].append(deepcopy(priors))
            results["threshold_true"].append(threshold_true)
            results["c"].append(c)
            results["opt"].append(deepcopy(partition_opt))
            results["greedy"].append(deepcopy(partition_greedy))
            results["acc_loss_opt"].append(acc_loss_opt.item())
            results["acc_loss_greedy"].append(acc_loss_greedy.item())
            results["ratio"].append(ratio.item())

In [ ]:
df = pd.DataFrame(results)
print(df.shape)
df.head()

In [ ]:
df_im = df.copy()
df_im["n"] = df_im["n"].astype(str)
df_im["c"] = df_im["c"].astype(str)
px.scatter(df_im, x="n", y="ratio", color="c", hover_data=["run"])

In [ ]:
px.scatter(df_im, x="c", y="ratio", color="n", hover_data=["run"])

In [ ]:
i_max = df["ratio"].argmax()
df_max = df.iloc[[i_max]]
df_max

In [ ]:
px.scatter(df, x="acc_loss_opt", y="ratio", color="acc_loss_greedy")#.update_traces(marker=dict(size=3))

In [ ]:
df_ = df[(df["acc_loss_greedy"]>0.9999) & (df["ratio"]>1)]
L = []
for i in range(len(df_)):
    L.append(len(df_.iloc[i]["greedy"]))
np.mean(L)

In [ ]:
px.scatter(df, x="threshold_true", y="acc_loss_opt")

In [ ]:
thresholds_ce = df_max["thresholds"].item()
priors_ce = df_max["priors"].item()

threshold_true_ce = df_max["threshold_true"].item()
c_ce = df_max["c"].item()

thresholds_ce[6] = thresholds_ce[7]
# partition_opt = find_partitions_optimal(X, thresholds_ce, priors_ce, threshold_true_ce, c_ce)
partition_greedy = find_partitions_greedy(X, thresholds_ce, priors_ce, threshold_true_ce, c_ce, True)

# acc_loss_opt = evaluate_system(X, partition_opt, thresholds_ce, priors_ce, threshold_true_ce, c_ce)
# acc_loss_greedy = evaluate_system(X, partition_greedy, thresholds_ce, priors_ce, threshold_true_ce, c_ce)
partition_opt = df_max["opt"].item()
partition_greedy = df_max["greedy"].item()

acc_loss_opt = df_max["acc_loss_opt"].item()
acc_loss_greedy = df_max["acc_loss_greedy"].item()

In [ ]:
print(f"c         : {c_ce:.4f}")
print(f"threshold true: {threshold_true_ce}")
display(pd.DataFrame({"Thresholds": thresholds_ce, "Priors": priors_ce}).round(6).T)

print("Greedy")
print("------")
print(f"Partition: {sorted(partition_greedy)}")
print(f"Acc Loss : {acc_loss_greedy:.7f}")
print()
print("Optimal")
print("-------")
print(f"Partition: {sorted(partition_opt)}")
print(f"Acc Loss : {acc_loss_opt:.7f}")
print()
print(f"Ratio: {(1 - acc_loss_opt) / (1 - acc_loss_greedy):.4f}")

In [ ]:
p = [0,1,2,3,4,5,6]
X_p = np.array([best_response(x, thresholds_ce[p], priors_ce[p], c_ce) for x in X])
px.scatter(x=X, y=X_p, labels={"x": "x", "y": "best response"}, title="All in 1")

In [ ]:
a, b = [0,1,2,3,4,5,6], [7]

acc_loss_a, thresholds_a, priors_a = evaluate_partition(X, a, thresholds_ce, priors_ce, threshold_true_ce, c_ce, True)
acc_loss_b, thresholds_b, priors_b = evaluate_partition(X, b, thresholds_ce, priors_ce, threshold_true_ce, c_ce, True)

lhs = acc_loss_a * np.sum(priors_ce[a]) + acc_loss_b * np.sum(priors_ce[b])
ab = sorted(a + b)

acc_loss_ab, thresholds_ab, priors_ab = evaluate_partition(X, ab, thresholds_ce, priors_ce, threshold_true_ce, c_ce, True)
rhs = acc_loss_ab * np.sum(priors_ce[ab])

print(f"    threshold true: {threshold_true_ce}")
print(f"                 c: {c_ce}")
print(f"                 a: {a}")
print(f"                 b: {b}")
print(f"   accuracy loss a: {acc_loss_a:.7f}")
print(f"   accuracy loss b: {acc_loss_b:.7f}")
print(f"  accuracy loss ab: {acc_loss_ab:.7f}")
print(f"               LHS: {lhs:.7f}")
print(f"               RHS: {rhs:.7f}")
print(f"            merge?: {lhs - rhs > -1e-9}")
print()

print(f"          priors a: {priors_a.round(4)}")
print(f"      thresholds a: {thresholds_a.round(4)}")
print(f"          priors b: {priors_b.round(4)}")
print(f"      thresholds b: {thresholds_b.round(4)}")
print(f"     thresholds ab: {thresholds_ab.round(4)}")
print(f"         priors ab: {priors_ab.round(4)}")

\begin{algorithm}[H]
\caption{Greedy Partition Merging}
\label{alg:greedy_partition}
\begin{algorithmic}[1]
\REQUIRE Data $X$, thresholds $t$, priors $p$, true threshold $t^*$, cost parameter $c$
\ENSURE A collection of disjoint index blocks forming a partition

\STATE Initialize partition dictionary $P$ so that each block contains a single index:
\[
P = \{ i \mapsto \{i\} : i = 1, \dots, |p| \}
\]

\STATE Initialize queue $Q$ with all unordered pairs of distinct block identifiers in $P$

\WHILE{$Q$ is not empty}
    \STATE Pop a pair $(a,b)$ from $Q$

    \IF{$a \notin P$ or $b \notin P$}
        \STATE \textbf{continue}
    \ENDIF

    \STATE Let $A \leftarrow P[a]$ and $B \leftarrow P[b]$

    \STATE Compute weighted loss before merging:
    \[
    L_{\mathrm{sep}} 
    = L(A)\cdot \sum_{i \in A} p_i
    + L(B)\cdot \sum_{i \in B} p_i
    \]
    where $L(\cdot)$ denotes \texttt{evaluate\_partition}

    \STATE Let $A \cup B$ denote the merged block

    \STATE Compute weighted loss after merging:
    \[
    L_{\mathrm{merge}}
    = L(A \cup B)\cdot \sum_{i \in A \cup B} p_i
    \]

    \IF{$L_{\mathrm{sep}} \ge L_{\mathrm{merge}} - \varepsilon$}
        \STATE Remove blocks $a$ and $b$ from $P$
        \STATE Remove from $Q$ all pairs involving $a$ or $b$

        \STATE Create new block $C = A \cup B$
        \STATE Assign new identifier $k$ and set $P[k] \leftarrow C$

        \FOR{each existing block identifier $d \in P$, $d \neq k$}
            \STATE Append $(k,d)$ to $Q$
        \ENDFOR
    \ENDIF
\ENDWHILE

\RETURN $\{ P[a] : a \in P \}$
\end{algorithmic}
\end{algorithm}